# Assignment 1 ml foundations: data prep

By Habib Rahal  

This notebook does the data preparation and feature engineering for the bank dataset. The goal of the campaign is to predict if a client would subscribe to a term deposit. we are going to work through the task 1 by 1.


## Task 1: Identifying the Prediction Target


In [1]:
import pandas as pd

DATA_PATH = "archive/bank-additional.csv" 


In [2]:
df = pd.read_csv(DATA_PATH, sep=';')

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
df.head()

Shape: (4119, 21)

Columns: ['age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'y']


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,30,blue-collar,married,basic.9y,no,yes,no,cellular,may,fri,...,2,999,0,nonexistent,-1.8,92.893,-46.2,1.313,5099.1,no
1,39,services,single,high.school,no,no,no,telephone,may,fri,...,4,999,0,nonexistent,1.1,93.994,-36.4,4.855,5191.0,no
2,25,services,married,high.school,no,yes,no,telephone,jun,wed,...,1,999,0,nonexistent,1.4,94.465,-41.8,4.962,5228.1,no
3,38,services,married,basic.9y,no,unknown,unknown,telephone,jun,fri,...,3,999,0,nonexistent,1.4,94.465,-41.8,4.959,5228.1,no
4,47,admin.,married,university.degree,no,yes,no,cellular,nov,mon,...,1,999,0,nonexistent,-0.1,93.200,-42.0,4.191,5195.8,no


The column that should be the prediction target is y. It indicates whether the client subscribed to a term deposit yes/no. thats how we predict whether the client subscribes given the information at contact time.

The two other columns could look like targets but should not be used:

- p of outcome: this is the outcome of the previous marketing campaign. It describes the past, not the current campaign outcome. So it is an input feature, not the target we want to predict.
- duration: call duration in seconds for the current contact. At the moment we would make a prediction (e.g. at the start of the call), we do not know the duration. Using it would leak information about the outcome, because longer calls often mean the client subscribed. Which means its a temporal leakage so it should not be the target and later we should not use it as a feature either. 

## Task 2: Data Loading and Exploration

Next we load the data  and look at it's structure: how many rows and columns, what types, and how the target is distributed. I also checked for missing values including unknowns and plot a some numerical and categorical variables to see what we are working with.

In [3]:
df = pd.read_csv(DATA_PATH, sep=';')

print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)
df.info()

Shape: (4119, 21)

Data types:
age                 int64
job                object
marital            object
education          object
default            object
housing            object
loan               object
contact            object
month              object
day_of_week        object
duration            int64
campaign            int64
pdays               int64
previous            int64
poutcome           object
emp.var.rate      float64
cons.price.idx    float64
cons.conf.idx     float64
euribor3m         float64
nr.employed       float64
y                  object
dtype: object
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4119 entries, 0 to 4118
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             4119 non-null   int64  
 1   job             4119 non-null   object 
 2   marital         4119 non-null   object 
 3   education       4119 non-null   object 
 4   default         4119 non-nu

In [4]:
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()

print("Numerical columns:", numeric_cols)
print("\nCategorical columns:", cat_cols)

print("\nSummary statistics (numerical):")
display(df[numeric_cols].describe())

print("\nSummary statistics (categorical):")
display(df[cat_cols].describe())

Numerical columns: ['age', 'duration', 'campaign', 'pdays', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']

Categorical columns: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome', 'y']

Summary statistics (numerical):


,age,duration,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed
count,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000
mean,40.113620,256.788055,2.537266,960.422190,0.190337,0.084972,93.579704,-40.499102,3.621356,5166.481695
std,10.313362,254.703736,2.568159,191.922786,0.541788,1.563114,0.579349,4.594578,1.733591,73.667904
min,18.000000,0.000000,1.000000,0.000000,0.000000,-3.400000,92.201000,-50.800000,0.635000,4963.600000
25%,32.000000,103.000000,1.000000,999.000000,0.000000,-1.800000,93.075000,-42.700000,1.334000,5099.100000
50%,38.000000,181.000000,2.000000,999.000000,0.000000,1.100000,93.749000,-41.800000,4.857000,5191.000000
75%,47.000000,317.000000,3.000000,999.000000,0.000000,1.400000,93.994000,-36.400000,4.961000,5228.100000
max,88.000000,3643.000000,35.000000,999.000000,6.000000,1.400000,94.767000,-26.900000,5.045000,5228.100000



Summary statistics (categorical):


,job,marital,education,default,housing,loan,contact,month,day_of_week,poutcome,y
count,4119,4119,4119,4119,4119,4119,4119,4119,4119,4119,4119
unique,12,4,8,3,3,3,2,10,5,3,2
top,admin.,married,university.degree,no,yes,no,cellular,may,thu,nonexistent,no
freq,1012,2509,1264,3315,2175,3349,2652,1378,860,3523,3668


In [5]:
print("Target (y) value counts:")
print(df["y"].value_counts())
print("\nProportions:")
print(df["y"].value_counts(normalize=True))

Target (y) value counts:
y
no     3668
yes     451
Name: count, dtype: int64

Proportions:
y
no     0.890507
yes    0.109493
Name: proportion, dtype: float64


The target is clearly imbalanced: most clients did not subscribe ("no"). So we will need to be careful later with metrics (accuracy can be misleading) and we might use resampling on the training set only.

In [ ]:
print("NaN counts per column:")
print(df.isna().sum())
print("\n'unknown' counts in categorical columns:")
for col in cat_cols:
    n = (df[col] == "unknown").sum()
    if n > 0:
        print(col, n)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes[0, 0].hist(df["age"], bins=30, edgecolor="black", alpha=0.7)
axes[0, 0].set_title("age")
axes[0, 0].set_xlabel("age")
axes[0, 1].hist(df["campaign"], bins=30, edgecolor="black", alpha=0.7)
axes[0, 1].set_title("campaign (number of contacts)")
axes[0, 1].set_xlabel("campaign")
df["job"].value_counts().plot(kind="bar", ax=axes[1, 0])
axes[1, 0].set_title("job")
axes[1, 0].tick_params(axis="x", rotation=45)
df["education"].value_counts().plot(kind="bar", ax=axes[1, 1])
axes[1, 1].set_title("education")
axes[1, 1].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

we stil need to consider the duration variable: "call duration in seconds". We already said in Task 1 that it is not available at prediction time because we don't know how long the call will last when we make the prediction. So it would cause data leakage if we use it as a feature. Ill exclude it when I build the model later.